# Machine learning

Nesta etapa, será desenvolvido um modelo de Machine Learning utilizando os dados do arquivo sinistros_sp_2025_limpo_eda. O objetivo é treinar um modelo de classificação capaz de identificar os acidentes de trânsito como fatais ou não fatais, buscando analisar os fatores associados à gravidade dos sinistros no município de São Paulo em 2025.

Ao longo do notebook, serão realizadas etapas de preparação dos dados, seleção das métricas e da variável-alvo, divisão dos dados em conjuntos de treino e teste, tratamento do desbalanceamento das classes, treinamento dos modelos e avaliação de seus resultados. Por fim, serão analisadas as métricas de desempenho para verificar a capacidade do modelo de classificar corretamente os sinistros.

## Importação das bibliotecas

In [1]:
# Manipulação de Dados
import pandas as pd
import numpy as np

# Visualização Gráfica
import matplotlib.pyplot as plt

#Separação dos dados: treino e teste
from sklearn.model_selection import train_test_split 

# Balanceamneto das classes
from imblearn.over_sampling import SMOTE

# Codificação de variáveis categóricas
from sklearn.preprocessing import OneHotEncoder

# Criação
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos de Aprendizado de Máquina
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor

# Métricas de avaliação dos modelos
from sklearn.metrics import (  #Métricas de Avaliação
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve
) 

## Carregamento dos dados

In [2]:
# Carregar a base
df_original = pd.read_csv(
    "../data/processed/sinistros_sp_2025_limpo_eda.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    low_memory=False
)
# Criar uma copia da base de dados
df = df_original.copy()

# mostrar os 5 primeiros registros
df.head()

,id_sinistro,tipo_registro,data_sinistro,ano_sinistro,mes_sinistro,dia_sinistro,hora_sinistro,ano_mes_sinistro,dia_da_semana,turno,...,tp_sinistro_choque,tp_sinistro_atrop_animal,tp_sinistro_capotamento,tp_sinistro_engavetamento,tp_sinistro_tombamento,tp_sinistro_outros,tp_sinistro_nao_disponivel,hora,latitude_num,longitude_num
0,2523341,NOTIFICACAO,01/01/2025,2025,1,1,03:11,2025/01,Quarta-feira,MADRUGADA,...,NaN,NaN,NaN,NaN,NaN,NaN,S,NaN,NaN,NaN
1,2467785,SINISTRO FATAL,01/01/2025,2025,1,1,02:40,2025/01,Quarta-feira,MADRUGADA,...,S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-23.775436541,-46.6817490556
2,2517646,NOTIFICACAO,01/01/2025,2025,1,1,01:15,2025/01,Quarta-feira,MADRUGADA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2524480,SINISTRO NAO FATAL,01/01/2025,2025,1,1,03:08,2025/01,Quarta-feira,MADRUGADA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-23.5078665,-46.4917972
4,2531534,SINISTRO NAO FATAL,02/01/2025,2025,1,2,20:40,2025/01,Quinta-feira,NOITE,...,NaN,NaN,NaN,NaN,NaN,S,NaN,NaN,-23.5755,-46.7129


## Definição da Variável-Alvo (fatal) e Remoção de Data Leakage

A variável-alvo do modelo será criada a partir da coluna `tipo_registro`. Inicialmente, serão mantidos apenas os registros classificados como **SINISTRO FATAL** e **SINISTRO NAO FATAL**, excluindo as notificações que não possuem uma classificação de gravidade definida. Em seguida, será criada a coluna **fatal**, que representará a gravidade do acidente de forma binária: 1 para sinistros **fatais** e 0 para sinistros **não fatais**. Por fim, será verificada a distribuição das classes para avaliar a quantidade de registros em cada categoria.

In [3]:
# Manter somente os registros classificados (fatal e não fatal)
df = df[
    df["tipo_registro"].isin([
        "SINISTRO FATAL",
        "SINISTRO NAO FATAL"
    ])
]

df["tipo_registro"].value_counts()

tipo_registro
SINISTRO NAO FATAL    25307
SINISTRO FATAL         1015
Name: count, dtype: int64

In [4]:
# Criar a variável-alvo (fatal)
df["fatal"] = (
    df["tipo_registro"] == "SINISTRO FATAL"
).astype(int)

# Conferir a distribuição do alvo
print(df["fatal"].value_counts())

fatal
0    25307
1     1015
Name: count, dtype: int64


## Pré-processamento

Nesta etapa, os dados serão preparados para o treinamento dos modelos, incluindo o tratamento de valores ausentes, a codificação de variáveis categóricas e a padronização das informações, quando necessário. O objetivo é garantir que os dados estejam adequados para os algoritmos de classificação.

### Tratamento de Data Leakage e remoção de colunas irrelevantes

Nesta etapa, serão removidas as colunas que possam comprometer a confiabilidade do modelo de classificação. Para evitar o data leakage (vazamento de dados), serão excluídas variáveis que contenham informações diretamente relacionadas à gravidade do acidente, pois podem revelar indiretamente a variável-alvo fatal. Também serão removidos identificadores únicos, informações constantes ou redundantes e colunas totalmente vazias. Além disso, serão descartadas as versões originais de latitude e longitude, ja que possuam suas versões numéricas disponíveis. O objetivo é reduzir informações desnecessárias e preparar o conjunto de dados para o treinamento do modelo.

In [5]:
# Remover identificadores e variáveis com vazamento de informação (Data Leakage)
colunas_remover = [
    # Alvo original e vazamento de dados
    "tipo_registro",
    "qtd_gravidade_fatal",
    "qtd_gravidade_grave",
    "qtd_gravidade_leve",
    "qtd_gravidade_ileso",
    "qtd_gravidade_nao_disponivel",

    # Identificação e informações constantes
    "id_sinistro",
    "cod_ibge",
    "municipio",
    "ano_sinistro",
    "regiao_administrativa",

    # Colunas vazias
    "hora",
    "tp_sinistro_colisao_traseira",

    # Colunas com informações redundantes
    "latitude_num",
    "longitude_num",
    "ano_mes_sinistro",
    "data_sinistro"
]

# Remoção das colunas
df = df.drop(
    columns=colunas_remover,
    errors="ignore"  # Evita um erro caso algum nome da lista não exista no DataFrame.
)

print("Dimensão final:", df.shape)

Dimensão final: (26322, 37)


### Tratamento das colunas com valores nulos

Os valores nulos serão tratados, prioritariamente, com base nas informações e orientações disponíveis no dicionário de dados. Nos casos em que não houver informações suficientes para determinar os valores correspondentes, serão adotadas técnicas de imputação de dados, considerando as características das variáveis e os objetivos do modelo, de modo a minimizar possíveis impactos no treinamento.


Nas colunas iniciadas por `qtd_*`, os valores nulos serão substituídos por **0** para viabilizar o treinamento do modelo. Essa substituição será adotada devido à ausência de informações que permitam determinar os valores reais correspondentes aos dados faltantes, sem pressupor que esses valores representam necessariamente uma quantidade igual a zero.

In [6]:
colunas_qtd = [
    col for col in df.columns
    if col.startswith("qtd_")
]

print(colunas_qtd)

['qtd_pedestre', 'qtd_bicicleta', 'qtd_motocicleta', 'qtd_automovel', 'qtd_onibus', 'qtd_caminhao', 'qtd_veic_outros', 'qtd_veic_nao_disponivel']


In [9]:
# Substituit os valores nulos por "0"
for col in colunas_qtd:
    df[col] = df[col].fillna(0)
    
    nulos = df[col].isna().sum()
    print(f"Coluna {col}: {nulos} valores nulos restantes")

Coluna qtd_pedestre: 0 valores nulos restantes
Coluna qtd_bicicleta: 0 valores nulos restantes
Coluna qtd_motocicleta: 0 valores nulos restantes
Coluna qtd_automovel: 0 valores nulos restantes
Coluna qtd_onibus: 0 valores nulos restantes
Coluna qtd_caminhao: 0 valores nulos restantes
Coluna qtd_veic_outros: 0 valores nulos restantes
Coluna qtd_veic_nao_disponivel: 0 valores nulos restantes


Na coluna `tipo_local` os valores nulos representam, de acordo com dicionário de dados, "NAO DISPONIVEL".

In [10]:
df["tipo_local"].value_counts()

tipo_local
PUBLICO           20599
NAO DISPONIVEL     5254
PRIVADO             112
Name: count, dtype: int64

In [12]:
# Substituir os valores nulos por "NAO DISPONIVEL"
df["tipo_local"] = df["tipo_local"].fillna("NAO DISPONIVEL")

# Mostrar os dados da coluna "tipo_local"
print(df["tipo_local"].value_counts())

tipo_local
PUBLICO           20599
NAO DISPONIVEL     5611
PRIVADO             112
Name: count, dtype: int64


As colunas `longitude`, `latitude`, `hora_sinistro`, `logradouro` e `numero_logradouro` apresentam valores nulos para os quais não foram identificadas informações complementares no dicionário de dados ou no site do Infosiga que permitam determinar os valores correspondentes. Dessa forma, os valores nulos serão substituídos pela expressão "NAO DISPONIVEL", a fim de padronizar o tratamento dos dados faltantes para o treinamento do modelo.

In [20]:
# Listar as colunas inadequadas
colunas_inadequadas = ["longitude","latitude","hora_sinistro","logradouro","numero_logradouro"]

# Substituir os valores nulos por "NAO DISPONIVEL"
for col in colunas_inadequadas:
    df[col] = df[col].fillna("NAO DISPONIVEL")

    # Confere se há valores nulos na colunas inadequadas
    print(df[col].isnull().sum())

0
0
0
0
0


### Binarização das variáveis de tipo de sinistro

Transformação das variáveis específicas, `tp_sinistro_...`, em valores binários, atribuindo 1 para indicar a **presença** do tipo de sinistro e 0 para indicar sua **ausência**, valores nulos, de acordo com o dicionário de dados. Essa conversão padroniza os dados categóricos para facilitar sua utilização no treinamento do modelo de Machine Learning.

In [21]:
# Lista todas as colunas relacionadas ao tipo de sinistro
colunas_sinistro = [
    col for col in df.columns
    if col.startswith("tp_sinistro_")
]

print(colunas_sinistro)

['tp_sinistro_primario', 'tp_sinistro_atrop_pedestre', 'tp_sinistro_atrop_vitima_fora_veic', 'tp_sinistro_colisao_frontal', 'tp_sinistro_colisao_lateral', 'tp_sinistro_colisao_transversal', 'tp_sinistro_colisao_outros', 'tp_sinistro_choque', 'tp_sinistro_atrop_animal', 'tp_sinistro_capotamento', 'tp_sinistro_engavetamento', 'tp_sinistro_tombamento', 'tp_sinistro_outros', 'tp_sinistro_nao_disponivel']


In [25]:
# Confere os valores de cada coluna
from IPython.display import display, Markdown

for col in colunas_sinistro:
    display(Markdown(f"### Coluna: `{col}`"))

    frequencia = (
        df[col]
        .value_counts(dropna=False)
        .head(10)
        .rename_axis('Valor')
        .reset_index(name='Quantidade')
    )

    display(frequencia)

### Coluna: `tp_sinistro_primario`

,Valor,Quantidade
0,COLISAO,18434
1,ATROPELAMENTO,3606
2,OUTROS,2627
3,CHOQUE,1127
4,NAO DISPONIVEL,528


### Coluna: `tp_sinistro_atrop_pedestre`

,Valor,Quantidade
0,NaN,22725
1,S,3597


### Coluna: `tp_sinistro_atrop_vitima_fora_veic`

...

,Valor,Quantidade
0,NaN,25177
1,S,1145


### Coluna: `tp_sinistro_outros`

,Valor,Quantidade
0,NaN,14658
1,S,11664


### Coluna: `tp_sinistro_nao_disponivel`

,Valor,Quantidade
0,NaN,25794
1,S,528


In [28]:
# Separa a coluna "tp_sinistro_primario"
colunas_indicadoras = [
    col for col in df.columns
    if col.startswith("tp_sinistro_")
    and col != "tp_sinistro_primario"
]

# Transforma S em 1 e campos vazios em 0, conforme o dicionário.
for col in colunas_indicadoras:
    df[col] = (
        df[col]
        .replace(r"^\s*$", pd.NA, regex=True) # transforma linhas vazias em nulos
        .fillna(0)
        .replace("S", 1)
        .astype(int)
    )

# Mostrar os dados das colunas "tp_sinistro_*"
display(df[colunas_indicadoras].head())

,tp_sinistro_atrop_pedestre,tp_sinistro_atrop_vitima_fora_veic,tp_sinistro_colisao_frontal,tp_sinistro_colisao_lateral,tp_sinistro_colisao_transversal,tp_sinistro_colisao_outros,tp_sinistro_choque,tp_sinistro_atrop_animal,tp_sinistro_capotamento,tp_sinistro_engavetamento,tp_sinistro_tombamento,tp_sinistro_outros,tp_sinistro_nao_disponivel
1,0,0,0,0,0,0,1,0,0,0,0,0,0
3,0,0,0,1,1,0,0,0,0,0,0,0,0
4,0,0,1,1,0,0,0,0,0,0,0,1,0
8,1,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,1,0,0


### Feature Engineering (engenharia de atributos)

Consiste na criação, transformação ou seleção de atributos a partir dos dados existentes, buscando destacar informações relevantes para o modelo. Neste projeto, essa etapa poderá ajudar a representar melhor características dos sinistros e suas possíveis relações com a gravidade dos acidentes.

In [29]:
# Converter a coluna "hora_sinistro" para formato de horário
hora = pd.to_datetime(
    df["hora_sinistro"],
    format="%H:%M",
    errors="coerce"
).dt.hour

# Mostra os dados de "hora sinistro"
print(df["hora_sinistro"].value_counts())

hora_sinistro
08:00    58
07:30    57
19:00    55
16:00    54
15:00    50
         ..
03:59     1
04:23     1
02:48     1
03:16     1
03:33     1
Name: count, Length: 1441, dtype: int64


In [32]:
df.info()

<class 'pandas.DataFrame'>
Index: 26322 entries, 1 to 42662
Data columns (total 37 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   mes_sinistro                        26322 non-null  int64 
 1   dia_sinistro                        26322 non-null  int64 
 2   hora_sinistro                       26322 non-null  str   
 3   dia_da_semana                       26322 non-null  str   
 4   turno                               26322 non-null  str   
 5   logradouro                          26322 non-null  str   
 6   numero_logradouro                   26322 non-null  str   
 7   tipo_via                            26322 non-null  str   
 8   tipo_local                          26322 non-null  str   
 9   latitude                            26322 non-null  object
 10  longitude                           26322 non-null  object
 11  administracao                       26322 non-null  str   
 12  conser

## Separação dos Dados em Treino e Teste

O conjunto de dados será dividido em duas partes: **80% para treinamento e 20% para teste**. Essa separação permite que os modelos de Machine Learning aprendam padrões a partir dos dados de treino e, posteriormente, sejam avaliados utilizando dados que não foram utilizados durante o treinamento.

A metodologia escolhida será o método *Holdout*, utilizando a função **train_test_split** do Scikit-learn. O parâmetro **test_size=0.20** define a proporção destinada ao teste, enquanto **random_state=42** garante a reprodutibilidade da divisão. Além disso, será utilizado **stratify=y** para manter aproximadamente a mesma proporção de sinistros fatais e não fatais nos dois conjuntos, considerando o desbalanceamento entre as classes.

In [33]:
# Separar as features do alvo
X = df.drop(columns=["fatal"])
y = df["fatal"]

# Dividir treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Conferir a distribuição
print("Treino:")
print(y_train.value_counts(normalize=True))

print("\nTeste:")
print(y_test.value_counts(normalize=True))

Treino:
fatal
0    0.961438
1    0.038562
Name: proportion, dtype: float64

Teste:
fatal
0    0.961443
1    0.038557
Name: proportion, dtype: float64


### Codificação de variáveis categóricas (One-Hot Encoding)

Nesta etapa, as variáveis categóricas serão convertidas em valores numéricos por meio de técnicas como o One-Hot Encoding, permitindo que os modelos de Machine Learning processem informações como turno, dia da semana e tipo de acidente.

In [34]:
# Criar o codificador
preprocessador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            make_column_selector(dtype_include=["object", "category", "bool"]), # Identificar as colunas categóricas
        )
    ],
    remainder="passthrough",
)

### Balanceamento das classes

Esta etapa busca lidar com o desbalanceamento entre as classes de sinistros fatais e não fatais. Como os acidentes fatais são menos frequentes, serão avaliadas técnicas para reduzir o impacto dessa diferença e melhorar a capacidade do modelo de identificar casos fatais.

## Modelos de classificação

### Logistic Regression (Regressão Logística)
A **Logistic Regression** é um algoritmo de aprendizado supervisionado usado principalmente para classificação, especialmente em problemas de classificação binária. Neste projeto, será utilizada para classificar os sinistros de trânsito como **fatais** ou **não fatais**.

In [4]:
# Construção do pipeline de configuração para o modelo de regressão logística
model_lr = Pipeline(
    steps=[
        ("pre_processamento", preprocessador),
        ("modelo", LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=42
            )
        )
    ]
)

# Treinar o pipeline do modelo com os dados de treino
model_lr.fit(X_train, y_train)

# Fazer previsões no conjunto de teste
y_pred_lr = model_lr.predict(x_test)

In [ ]:
# Exibe os resultados do modelo
print(classification_report(y_test, y_pred_lr, target_names=["Não fatal", "Fatal"]))

In [ ]:
# Probabilidade da classe fatal
y_prob_lr = model_lr.predict_proba(x_test)[:,1]

# Calcular os pontos da curva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)

# Gráfico da curva ROC
plt.plot(fpr_lr, tpr_lr)
plt.title("ROC curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

# Mostrar a porcentagem da área sobre a curva
print("AUC:", roc_auc_score(y_test, y_prob_lr))

A curva ROC é uma métrica importante no modelo de classificação. O gráfico mostra o desempenho do modelo em diferentes lineares de decisão.

- false positive rate, representa a taxa de falsos positivos, ou seja, quantos acidentes **não grave** foram classificadas como **grave**.
- true positive rate, representa o mesmo que o recall, ou seja, quantos **acidentes grave** o modelo conseguiu identificar.

Cada ponto da curva representa um limite na decisão do modelo, quanto mais a curva se aproxima do canto esquerdo melhor é o modelo (alto recall e baixo erro).

In [ ]:
# Calcular os pontos da curva Precision_Recall
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_prob_lr)

# Gráfico da curva Precision-Recall
plt.plot(recall_lr, precision_lr)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

A curva precision-recall é outra métrica importante, principalmente em classes desbalanceadas como é o caso. O gráfico mostra o trade-off entre essas duas métricas, ou seja, o equilíbrio entre elas: à medida que uma aumenta, a outra diminui.

- Recall, representa quantos **acidentes graves** o modelo consegue identificar.
- Precision, representa a precisão de quantos **acidentes graves** identificados são realmente **grave**.

### Matriz de Confusão

A matriz de confusão será utilizada para avaliar o desempenho dos modelos de classificação, comparando as classes reais com as previstas. Essa análise permitirá identificar quantos sinistros fatais e não fatais foram classificados corretamente ou incorretamente, destacando possíveis dificuldades na identificação de acidentes fatais.

In [ ]:
cm_lr = confusion_matrix(y, predictions)

disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm)

disp_lr.plot()

plt.show()

### Random Forest (Floresta Aleatória)
O **Random Forest** é um algoritmo de aprendizado supervisionado usado principalmente para classificação, ele combina várias árvores de decisão para realizar uma classificação. Essa abordagem pode identificar relações complexas entre as métricas dos acidentes, contribuindo para analisar quais fatores estão associados à ocorrência de sinistros fatais.

In [6]:

model_rfc = Pipeline(
    steps=[
        ("pre_processamento", preprocessador),
        ("modelo", RandomForestClassifier(
                    n_estimators=200,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1
            )
        )
    ]
)

# Treinar o pipeline do modelo com os dados de treino
model_rfc.fit(X_train, y_train)

# Fazer previsões no conjunto de teste
y_pred_rfc = model_rfc.predict(x_test)

In [ ]:
# Exibe os resultados do modelo
print(classification_report(y_test, y_pred_rfc, target_names=["Não fatal", "Fatal"]))

In [ ]:
# Probabilidade da classe fatal
y_prob_rfc = model_rfc.predict_proba(x_test)[:,1]

# Calcular os pontos da curva ROC
fpr_rfc, tpr_rfc, _ = roc_curve(y_test, y_prob_rfc)

# Gráfico da curva ROC
plt.plot(fpr_rfc, tpr_rfc)
plt.title("ROC curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

# Mostrar a porcentagem da área sobre a curva
print("AUC:", roc_auc_score(y_test, y_prob_rfc))

In [ ]:
# Calcular os pontos da curva Precision_Recall
precision_rfc, recall_rfc, _ = precision_recall_curve(y_test, y_prob_rfc)

# Gráfico da curva Precision-Recall
plt.plot(recall_rfc, precision_rfc)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

#### Matriz de Confusão

In [ ]:
cm_rfc = confusion_matrix(y, predictions)

disp_rfc = ConfusionMatrixDisplay(confusion_matrix=cm)

disp_rfc.plot()

plt.show()

### Decision Tree (Árvore de Decisão)

A **Decision Tree** é um algoritmo e aprendizado supervisionado usado tanto para classificação quanto para regressão, utiliza regras e condições para classificar os dados. Neste projeto, poderá identificar padrões entre as métricas dos acidentes e a gravidade dos sinistros. Uma de suas principais vantagens é a facilidade de interpretação dos critérios utilizados pelo modelo.

In [ ]:

model_dtc = DecisionTreeClassifier()
Pipeline(
    steps=[
        ("pre_processamento", preprocessador),
        ("modelo", DecisionTreeClassifier(
                    class_weight="balanced",
                    random_state=42
            )
        )
    ]
)

# Treinar o pipeline do modelo com os dados de treino
model_dtc.fit(X_train, y_train)

# Fazer previsões no conjunto de teste
y_pred_dtc = model_dtc.predict(x_test)

In [ ]:
# Exibe os resultados do modelo
print(classification_report(y_test, y_pred_dtc, target_names=["Não fatal", "Fatal"]))

In [ ]:
# Probabilidade da classe fatal
y_prob_dtc = model_dtc.predict_proba(x_test)[:,1]

# Calcular os pontos da curva ROC
fpr_dtc, tpr_dtc, _ = roc_curve(y_test, y_prob_dtc)

# Gráfico da curva ROC
plt.plot(fpr_dtc, tpr_dtc)
plt.title("ROC curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

# Mostrar a porcentagem da área sobre a curva
print("AUC:", roc_auc_score(y_test, y_prob_dtc))

In [ ]:
# Calcular os pontos da curva Precision_Recall
precision_dtc, recall_dtc, _ = precision_recall_curve(y_test, y_prob_dtc)

# Gráfico da curva Precision-Recall
plt.plot(recall_dtc, precision_dtc)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

#### Matriz de Confusão

In [ ]:
cm_dtc = confusion_matrix(y, predictions)

disp_dtc = ConfusionMatrixDisplay(confusion_matrix=cm)

disp_dtc.plot()

plt.show()

## Comparação dos modelos e validação cruzada

Nesta etapa, os resultados da Regressão Logística, Random Forest e Árvore de Decisão serão comparados utilizando métricas de avaliação, como precision, recall, F1-score e matriz de confusão. O objetivo é identificar qual modelo apresenta o desempenho mais adequado para classificar os sinistros como fatais ou não fatais.

- Accuracy → quantas previsões acertou no total;
- Precision → quando previu determinada classe, quantas vezes estava correto;
- Recall → quantos casos daquela classe conseguiu identificar;
- F1-Score → equilíbrio entre Precision e Recall.
- Matriz de Confusão → mostra detalhadamente os acertos e erros por classe.

A validação cruzada consiste em dividir os dados de treinamento em diferentes partes (folds) para treinar e avaliar o modelo em combinações distintas. Essa técnica permite analisar a estabilidade do desempenho e reduz a dependência de uma única divisão dos dados.

Com dados desbalanceados, um único split treino/teste pode ser "sortudo" ou "azarado". Por exemplo:

Se por acaso os fatais ficaram mais concentrados no treino → modelo aprende melhor.                                                                   
Se ficaram mais no teste → métricas ruins

O **k-fold cross-validation** repetiria o processo 5 ou 10 vezes com divisões diferentes e mostraria a variação real da performance. Sem isso, as métricas apresentadas são uma foto de um único momento, potencialmente não representativa.

### Comparação das métricas

In [ ]:
report_random_forest = classification_report()
report_logistic_regression = classification_report()
report_decision_tree = classification_report()

# Criar tabela dinâmica para comparação.
resultados = pd.DataFrame({
    "Modelo": [
        "Random Forest",
        "Logistic Regression",
        "Decision Tree"
    ],
    "Tipo": [
        "Supervisionado",
        "Supervisionado",
        "Supervisionado"
    ],
    "Precision": [
        report_random_forest["1"]["precision"],
        report_logistic_regression["1"]["precision"],
        report_decision_tree["1"]["precision"]
    ],
    "Recall": [
        report_random_forest["1"]["recall"],
        report_logistic_regression["1"]["recall"],
        report_decision_tree["1"]["recall"]
    ],
    "F1-Score": [
        report_random_forest["1"]["f1-score"],
        report_logistic_regression["1"]["f1-score"],
        report_decision_tree["1"]["f1-score"]
    ]
})

# Mostrar tabela. 
resultados.round(2) # "round" limita casas decimais.

In [ ]:
# Gráfico de comparação
resultados.set_index("Modelo")[["Precision", "Recall", "F1-Score"]].plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Comparação entre modelos")
plt.ylabel("Score")
plt.ylim(0, 1)

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    fpr_log,
    tpr_log,
    label=f"Regressão Logística (AUC = {auc:.4f})"
)

plt.plot(
    fpr_log,
    tpr_log,
    label=f"Regressão Logística (AUC = {auc:.4f})"
)

plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest (AUC = {auc_rf:.4f})"
)

# Linha de referência
plt.plot([0, 1], [0, 1], linestyle="--", label="Referência aleatória")

plt.title(
    "Curva ROC dos Modelos de Machine Learning\n"
    "São Paulo Capital (2025)"
)

plt.xlabel("Taxa de Falsos Positivos")
plt.ylabel("Taxa de Verdadeiros Positivos")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "23_curva_roc_modelos_machine_learning_2025.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## análise de erros (opcional)

Nesta etapa, serão investigadas as previsões incorretas dos modelos para identificar padrões e possíveis limitações na classificação dos sinistros. O objetivo será compreender em quais situações o modelo apresenta maior dificuldade, especialmente na identificação de acidentes fatais, contribuindo para interpretar os resultados e identificar oportunidades de melhoria.

## Conclusão